In [1]:
import os

# Move up one level to set the working directory to the repo root
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import pandas as pd

from src.data_processing import extract_game_info, extract_seed_value

In [15]:
# Define function to determine gender based on TeamID1
def determine_gender(team_id):
    if str(team_id).startswith("1"):  # Men's teams start with 1
        return "Men"
    elif str(team_id).startswith("3"):  # Women's teams start with 3
        return "Women"
    else:
        return None  # Handle unexpected cases


# Function to merge results_df with seeds_df and tourney_round_lookup
def process_results(results_df, seeds_df, tourney_round_lookup):
    results_df = results_df.merge(
        seeds_df,
        left_on=["Season", "WTeamID"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T1"),
    )
    results_df = results_df.merge(
        seeds_df,
        left_on=["Season", "LTeamID"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T2"),
    )

    results_df["WSeed"] = results_df["Seed"]
    results_df["LSeed"] = results_df["Seed_T2"]

    # Ensure StrongSeed is alphabetically first, and WeakSeed is second
    results_df["StrongSeed"] = results_df[["WSeed", "LSeed"]].min(axis=1)
    results_df["WeakSeed"] = results_df[["WSeed", "LSeed"]].max(axis=1)

    results_df = results_df.merge(
        tourney_round_lookup,
        left_on=["StrongSeed", "WeakSeed"],
        right_on=["StrongSeed", "WeakSeed"],
        how="left",
    )

    # Drop unnecessary columns and return
    results_df = results_df[
        [
            "Season",
            "DayNum",
            "WTeamID",
            "WSeed",
            "WScore",
            "LTeamID",
            "LSeed",
            "LScore",
            "WLoc",
            "NumOT",
            "Round",
            "Slot",
        ]
    ]

    return results_df


# Function to merge sub_df with seeds_df and tourney_round_lookup
def process_submission(sub_df, seeds_df, tourney_round_lookup):
    sub_df = sub_df.merge(
        seeds_df,
        left_on=["Season", "TeamID1"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T1"),
    )
    sub_df = sub_df.merge(
        seeds_df,
        left_on=["Season", "TeamID2"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T2"),
    )

    sub_df["Seed1"] = sub_df["Seed"].fillna("")
    sub_df["Seed2"] = sub_df["Seed_T2"].fillna("")

    # Ensure StrongSeed is alphabetically first, and WeakSeed is second
    sub_df["StrongSeed"] = sub_df[["Seed1", "Seed2"]].min(axis=1)
    sub_df["WeakSeed"] = sub_df[["Seed1", "Seed2"]].max(axis=1)

    sub_df = sub_df.merge(
        tourney_round_lookup,
        left_on=["StrongSeed", "WeakSeed"],
        right_on=["StrongSeed", "WeakSeed"],
        how="left",
    )

    # Drop unnecessary columns and return
    sub_df = sub_df[
        [
            "ID",
            "Pred",
            "Season",
            "TeamID1",
            "Seed1",
            "TeamID2",
            "Seed2",
            "Round",
            "Slot",
        ]
    ]

    return sub_df

In [16]:
# Load data
m_seed = pd.read_csv(r"data\kaggle\MNCAATourneySeeds.csv")
w_seed = pd.read_csv(r"data\kaggle\WNCAATourneySeeds.csv")
m_results = pd.read_csv(r"data\kaggle\MNCAATourneyCompactResults.csv")
w_results = pd.read_csv(r"data\kaggle\WNCAATourneyCompactResults.csv")
submission_df = pd.read_csv(r"data\kaggle\SampleSubmissionStage1.csv")
tourney_round_lookup = pd.read_csv(r"data\tourney_round_lookup.csv")

In [17]:
# Extract numeric seed values
m_seed["SeedValue"] = m_seed["Seed"].apply(extract_seed_value)
w_seed["SeedValue"] = w_seed["Seed"].apply(extract_seed_value)

# Merge results_df with seeds_df and tourney_round_lookup
m_results = process_results(m_results, m_seed, tourney_round_lookup)
w_results = process_results(w_results, w_seed, tourney_round_lookup)

# Extract game info
submission_df[["Season", "TeamID1", "TeamID2"]] = (
    submission_df["ID"].apply(extract_game_info).apply(pd.Series)
)

# Assign Gender column
submission_df["Gender"] = submission_df["TeamID1"].apply(determine_gender)

# Split into men's and women's submission dataframes
m_submission = submission_df[submission_df["Gender"] == "Men"].copy()
w_submission = submission_df[submission_df["Gender"] == "Women"].copy()

# Merge sub_df with seeds_df and tourney_round_lookup
m_submission = process_submission(m_submission, m_seed, tourney_round_lookup)
w_submission = process_submission(w_submission, w_seed, tourney_round_lookup)

In [18]:
m_results.head()

,Season,DayNum,WTeamID,WSeed,WScore,LTeamID,LSeed,LScore,WLoc,NumOT,Round,Slot
0,1985,136,1116,X09,63,1234,X08,54,N,0,1.0,R1X8
1,1985,136,1120,Z11,59,1345,Z06,58,N,0,1.0,R1Z6
2,1985,136,1207,W01,68,1250,W16,43,N,0,1.0,R1W1
3,1985,136,1229,Y09,58,1425,Y08,55,N,0,1.0,R1Y8
4,1985,136,1242,Z03,49,1325,Z14,38,N,0,1.0,R1Z3


In [19]:
m_submission.head()

,ID,Pred,Season,TeamID1,Seed1,TeamID2,Seed2,Round,Slot
0,2021_1101_1102,0.5,2021,1101,W14,1102,,NaN,NaN
1,2021_1101_1103,0.5,2021,1101,W14,1103,,NaN,NaN
2,2021_1101_1104,0.5,2021,1101,W14,1104,W02,3.0,R3W2
3,2021_1101_1105,0.5,2021,1101,W14,1105,,NaN,NaN
4,2021_1101_1106,0.5,2021,1101,W14,1106,,NaN,NaN


In [20]:
w_results.head()

,Season,DayNum,WTeamID,WSeed,WScore,LTeamID,LSeed,LScore,WLoc,NumOT,Round,Slot
0,1998,137,3104,X02,94,3422,X15,46,H,0,1.0,R1X2
1,1998,137,3112,W03,75,3365,W14,63,H,0,1.0,R1W3
2,1998,137,3163,W02,93,3193,W15,52,H,0,1.0,R1W2
3,1998,137,3198,Y07,59,3266,Y10,45,H,0,1.0,R1Y7
4,1998,137,3203,W10,74,3208,W07,72,A,0,1.0,R1W7


In [21]:
w_submission.head()

,ID,Pred,Season,TeamID1,Seed1,TeamID2,Seed2,Round,Slot
0,2021_3101_3102,0.5,2021,3101,,3102,,NaN,NaN
1,2021_3101_3103,0.5,2021,3101,,3103,,NaN,NaN
2,2021_3101_3104,0.5,2021,3101,,3104,X07,NaN,NaN
3,2021_3101_3105,0.5,2021,3101,,3105,,NaN,NaN
4,2021_3101_3106,0.5,2021,3101,,3106,,NaN,NaN


In [27]:
import pandas as pd
import re


# Function to extract seed value from seed string
def extract_seed_value(seed_str: str) -> int:
    """
    Extracts the numeric seed value from an NCAA tournament seed string.

    Args:
        seed_str (str): The seed string (e.g., 'W01', 'Y12b').

    Returns:
        int: The extracted seed value, or 16 if extraction fails.
    """
    try:
        match = re.search(r"\d+", seed_str)
        if match:
            return int(match.group())
        else:
            return 16  # Default seed value for unselected teams/errors
    except (ValueError, TypeError):
        return 16  # Default for unexpected cases


# Filter for Round 1 games
m_round1 = m_results[m_results["Round"] == 1].copy()
w_round1 = w_results[w_results["Round"] == 1].copy()

# Extract numeric seed values
m_round1["WSeedValue"] = m_round1["WSeed"].apply(extract_seed_value)
m_round1["LSeedValue"] = m_round1["LSeed"].apply(extract_seed_value)
w_round1["WSeedValue"] = w_round1["WSeed"].apply(extract_seed_value)
w_round1["LSeedValue"] = w_round1["LSeed"].apply(extract_seed_value)

# Determine StrongSeed and WeakSeed
m_round1["StrongSeed"] = m_round1[["WSeedValue", "LSeedValue"]].min(axis=1)
m_round1["WeakSeed"] = m_round1[["WSeedValue", "LSeedValue"]].max(axis=1)
w_round1["StrongSeed"] = w_round1[["WSeedValue", "LSeedValue"]].min(axis=1)
w_round1["WeakSeed"] = w_round1[["WSeedValue", "LSeedValue"]].max(axis=1)

# Identify if the StrongSeed won
m_round1["StrongSeedWon"] = m_round1["StrongSeed"] == m_round1["WSeedValue"]
w_round1["StrongSeedWon"] = w_round1["StrongSeed"] == w_round1["WSeedValue"]


# Calculate win percentages for each seed matchup
def compute_seed_win_percentage(df):
    matchup_counts = (
        df.groupby(["StrongSeed", "WeakSeed"]).size().reset_index(name="TotalGames")
    )
    win_counts = (
        df[df["StrongSeedWon"]]
        .groupby(["StrongSeed", "WeakSeed"])
        .size()
        .reset_index(name="Wins")
    )
    merged = pd.merge(
        matchup_counts, win_counts, on=["StrongSeed", "WeakSeed"], how="left"
    ).fillna(0)
    merged["WinPercentage"] = (merged["Wins"] / merged["TotalGames"]) * 100
    return merged.sort_values(by=["StrongSeed", "WeakSeed"])


m_seed_win_percent = compute_seed_win_percentage(m_round1)
w_seed_win_percent = compute_seed_win_percentage(w_round1)

# Display results
print("Men's Tournament Seed Win Percentages:")
print(m_seed_win_percent)
print("\nWomen's Tournament Seed Win Percentages:")
print(w_seed_win_percent)

Men's Tournament Seed Win Percentages:
   StrongSeed  WeakSeed  TotalGames  Wins  WinPercentage
0           1        16         120   119      99.166667
1           2        15         156   145      92.948718
2           3        14         155   132      85.161290
3           4        13         155   123      79.354839
4           5        12         152    98      64.473684
5           6        11         138    86      62.318841
6           7        10         153    94      61.437908
7           8         9         156    75      48.076923

Women's Tournament Seed Win Percentages:
   StrongSeed  WeakSeed  TotalGames  Wins  WinPercentage
0           1        16          98    97      98.979592
1           2        15         104   104     100.000000
2           3        14         104   104     100.000000
3           4        13         104    98      94.230769
4           5        12         103    81      78.640777
5           6        11          99    65      65.656566
6      